# 🎯 Notebook 03: CO-STAR Prompt Engineering Framework

**Time:** 30 minutes  
**Goal:** Master structured prompt engineering with the CO-STAR framework

## What is CO-STAR?

CO-STAR is a systematic framework for writing effective prompts:

- **C**ontext - Background information
- **O**bjective - What you want to achieve
- **S**tyle - Communication style (formal, casual, technical)
- **T**one - Emotional tone (friendly, authoritative, humorous)
- **A**udience - Who will read the response
- **R**esponse Format - Expected output structure (JSON, markdown, list)

## Why CO-STAR?

Without structure, prompts can be:
- Vague and ambiguous
- Missing crucial context
- Producing inconsistent results
- Hard to reproduce or improve

CO-STAR gives you a **repeatable framework** for crafting production-quality prompts.

Let's master it! 🚀

**Prerequisites:** Notebook 02 completed

In [1]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, format_response, append_to_reflection
from src.prompt_templates import COSTARTemplate
from src.config import PATH

print("=" * 60)
print("NOTEBOOK 03: CO-STAR FRAMEWORK")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

print()
print("✓ Ready to learn CO-STAR!")
print()

NOTEBOOK 03: CO-STAR FRAMEWORK

Configuration loaded: Path A

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5

✓ Ready to learn CO-STAR!



---
## 📚 Understanding Each Component

Let's break down each component of CO-STAR with examples.

### 🅲 Context - Setting the Stage

**Purpose:** Provide background information the LLM needs to understand the task.

**Examples:**
- ❌ Bad: "Write some code"
- ✅ Good: "You are helping a junior developer learn Python. They know basic syntax but struggle with list comprehensions."

**When to use:**
- Domain-specific tasks
- When the LLM needs background knowledge
- Role-playing scenarios
- Tasks requiring specific expertise

### 🅾 Objective - Define Clear Goals

**Purpose:** Explicitly state what you want to achieve.

**Examples:**
- ❌ Bad: "Help me with data"
- ✅ Good: "Analyze this sales data and identify the top 3 trends affecting Q4 revenue."

**Tips:**
- Be specific and measurable
- Use action verbs (analyze, summarize, create, explain)
- State success criteria when relevant

### 🆂 Style - Communication Approach

**Purpose:** Define how the response should be written.

**Options:**
- Professional / Casual
- Technical / Non-technical
- Academic / Conversational
- Concise / Detailed
- Creative / Analytical

**Examples:**
- "Write in a professional, business-appropriate style"
- "Use casual, conversational language suitable for a blog post"
- "Adopt an academic style with proper citations"

### 🆃 Tone - Emotional Character

**Purpose:** Set the emotional quality of the response.

**Options:**
- Friendly / Formal
- Encouraging / Critical
- Humorous / Serious
- Authoritative / Exploratory
- Empathetic / Neutral

**Examples:**
- "Use a friendly, encouraging tone"
- "Maintain a neutral, objective tone"
- "Be authoritative but not condescending"

### 🅰 Audience - Who's Reading?

**Purpose:** Tailor complexity and terminology to the reader.

**Consider:**
- Expertise level (beginner, intermediate, expert)
- Role (student, executive, engineer)
- Age group (if relevant)
- Cultural context

**Examples:**
- "Explain to a 5-year-old"
- "Write for experienced data scientists"
- "Target C-level executives with limited technical background"

### 🆁 Response Format - Structure Matters

**Purpose:** Specify exactly how you want the output structured.

**Common formats:**
- Plain text / Paragraphs
- JSON / XML
- Markdown with headers
- Numbered lists / Bullet points
- Table format
- Code blocks

**Examples:**
- "Respond in valid JSON with keys: 'summary', 'recommendations', 'next_steps'"
- "Use markdown with H2 headers for each section"
- "Provide a numbered list of exactly 5 items"

---
## 🧪 Experiment 1: Component-by-Component Comparison

Let's see the impact of adding each CO-STAR component one at a time.

In [2]:
# Progressive CO-STAR Build-up
print("=" * 60)
print("EXPERIMENT 1: Building CO-STAR Step-by-Step")
print("=" * 60)
print()

base_question = "How do I prepare for a technical interview?"

prompts = {
    "No Structure": base_question,
    
    "+ Context": """Context: You're a senior software engineer who has conducted hundreds of technical interviews.
    
How do I prepare for a technical interview?""",
    
    "+ Objective": """Context: You're a senior software engineer who has conducted hundreds of technical interviews.
Objective: Provide actionable preparation steps that cover both technical and behavioral aspects.

How do I prepare for a technical interview?""",
    
    "+ Style": """Context: You're a senior software engineer who has conducted hundreds of technical interviews.
Objective: Provide actionable preparation steps that cover both technical and behavioral aspects.
Style: Professional but approachable, like a mentor giving advice.

How do I prepare for a technical interview?""",
    
    "+ Tone": """Context: You're a senior software engineer who has conducted hundreds of technical interviews.
Objective: Provide actionable preparation steps that cover both technical and behavioral aspects.
Style: Professional but approachable, like a mentor giving advice.
Tone: Encouraging and confidence-building.

How do I prepare for a technical interview?""",
    
    "+ Audience": """Context: You're a senior software engineer who has conducted hundreds of technical interviews.
Objective: Provide actionable preparation steps that cover both technical and behavioral aspects.
Style: Professional but approachable, like a mentor giving advice.
Tone: Encouraging and confidence-building.
Audience: Mid-level developer with 2-3 years of experience preparing for a senior role.

How do I prepare for a technical interview?""",
    
    "Full CO-STAR": """Context: You're a senior software engineer who has conducted hundreds of technical interviews.
Objective: Provide actionable preparation steps that cover both technical and behavioral aspects.
Style: Professional but approachable, like a mentor giving advice.
Tone: Encouraging and confidence-building.
Audience: Mid-level developer with 2-3 years of experience preparing for a senior role.
Response Format: Provide 5 specific steps, each with a brief explanation (2-3 sentences).

How do I prepare for a technical interview?"""
}

print("Testing different levels of CO-STAR structure...")
print()

results = {}

for label, prompt in prompts.items():
    print(f"📝 {label}")
    print("-" * 60)
    
    response = client.generate(
        prompt=prompt,
        temperature=0.7,
        max_tokens=300
    )
    
    if "error" not in response:
        print(f"Tokens: {response['usage']['input_tokens']} in, {response['usage']['output_tokens']} out")
        print()
        print("Response preview:")
        print(response['content'][:200] + "..." if len(response['content']) > 200 else response['content'])
        results[label] = response['content']
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

print("=" * 60)
print("💡 OBSERVATION")
print("=" * 60)
print()
print("Notice how each additional component makes the response:")
print("  • More focused and relevant")
print("  • Better structured")
print("  • More appropriate for the audience")
print("  • Easier to use directly")
print()
print("Full CO-STAR = Maximum control and quality!")
print()

EXPERIMENT 1: Building CO-STAR Step-by-Step

Testing different levels of CO-STAR structure...

📝 No Structure
------------------------------------------------------------
Tokens: 16 in, 300 out

Response preview:
# Technical Interview Prep Guide

## 1. **Master the Fundamentals**
- **Data Structures**: Arrays, linked lists, trees, graphs, hash tables, stacks, queues
- **Algorithms**: Sorting, searching, recurs...


📝 + Context
------------------------------------------------------------
Tokens: 33 in, 300 out

Response preview:
# Technical Interview Preparation Guide

## 1. **Fundamentals First**
- **Data Structures**: Arrays, LinkedLists, Trees, Graphs, HashMaps, Stacks, Queues, Heaps
- **Algorithms**: Sorting, searching, r...


📝 + Objective
------------------------------------------------------------
Tokens: 51 in, 300 out

Response preview:
# Technical Interview Preparation Guide

## Technical Preparation

### 1. **Data Structures & Algorithms (4-6 weeks)**
- **Core topics to maste

---
## 🎨 Experiment 2: Using the CO-STAR Template Builder

We have a helper class that makes building CO-STAR prompts easy!

In [3]:
# Using COSTARTemplate Builder
print("=" * 60)
print("EXPERIMENT 2: CO-STAR Template Builder")
print("=" * 60)
print()

# Build a CO-STAR prompt using the template
costar_prompt = COSTARTemplate.build(
    context="You are a data scientist analyzing customer churn for an e-commerce company.",
    objective="Identify the top 3 factors driving customer churn and suggest mitigation strategies.",
    style="analytical and data-driven",
    tone="objective and professional",
    audience="company executives with limited data science background",
    response_format="structured report with sections: Key Findings, Root Causes, Recommendations"
)

print("Generated CO-STAR Prompt:")
print("=" * 60)
print(costar_prompt)
print("=" * 60)
print()

# Test it
print("Testing this prompt...")
print()

response = client.generate(
    prompt=costar_prompt,
    temperature=0.7,
    max_tokens=400
)

if "error" not in response:
    print("Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    print()
    print(f"Tokens: {response['usage']['input_tokens']} in, {response['usage']['output_tokens']} out")
    tracker.add_call(response)
else:
    print(f"Error: {response['error']}")

print()

EXPERIMENT 2: CO-STAR Template Builder

Generated CO-STAR Prompt:
# Context
You are a data scientist analyzing customer churn for an e-commerce company.

# Objective
Identify the top 3 factors driving customer churn and suggest mitigation strategies.

# Style
analytical and data-driven

# Tone
objective and professional

# Audience
company executives with limited data science background

# Response Format
structured report with sections: Key Findings, Root Causes, Recommendations


Testing this prompt...

Response:
# Customer Churn Analysis Report

## Executive Summary
This report presents findings from our analysis of customer churn patterns, identifying the primary drivers of customer attrition and actionable strategies to improve retention rates.

---

## Key Findings

Based on comprehensive data analysis, three factors demonstrate the strongest correlation with customer churn:

### 1. **Low Purchase Frequency (Impact Score: 8.5/10)**
Customers making fewer than 2 purchases within t

---
## 🏗️ Experiment 3: System Prompt vs User Prompt for CO-STAR

You can put CO-STAR components in either:
1. **System Prompt** - Sets overall behavior (C, S, T, A, R)
2. **User Prompt** - Contains specific request (O)

Let's compare approaches!

In [4]:
# System vs User Prompt Placement
print("=" * 60)
print("EXPERIMENT 3: System vs User Prompt Placement")
print("=" * 60)
print()

# Approach 1: Everything in user prompt
print("APPROACH 1: All CO-STAR in User Prompt")
print("-" * 60)

user_only_prompt = COSTARTemplate.build(
    context="You are a fitness coach",
    objective="Create a beginner workout plan",
    style="encouraging and supportive",
    tone="friendly and motivating",
    audience="complete beginner who has never exercised",
    response_format="3 exercises with sets/reps"
)

response1 = client.generate(
    prompt=user_only_prompt,
    temperature=0.7,
    max_tokens=250
)

if "error" not in response1:
    print(response1['content'])
    tracker.add_call(response1)
print()
print()

# Approach 2: C, S, T, A, R in system; O in user
print("APPROACH 2: Split - System (C,S,T,A,R) + User (O)")
print("-" * 60)

system_prompt = COSTARTemplate.build_system(
    style="encouraging and supportive",
    tone="friendly and motivating",
    response_format="3 exercises with sets/reps"
)

# Add context manually
system_prompt = f"You are a fitness coach. {system_prompt}\nAudience: Complete beginner who has never exercised."

user_prompt = "Create a beginner workout plan."

response2 = client.generate(
    prompt=user_prompt,
    system=system_prompt,
    temperature=0.7,
    max_tokens=250
)

if "error" not in response2:
    print(response2['content'])
    tracker.add_call(response2)
print()
print()

print("=" * 60)
print("💡 WHICH APPROACH TO USE?")
print("=" * 60)
print()
print("Approach 1 (All in User):")
print("  ✓ Simpler - everything in one place")
print("  ✓ Easier to track full prompt")
print("  ✗ More tokens per request (higher cost)")
print()
print("Approach 2 (Split System/User):")
print("  ✓ Reusable system prompt across many requests")
print("  ✓ Lower cost (S,T,A,R set once)")
print("  ✓ Cleaner user prompts")
print("  ✗ Slightly more complex to manage")
print()
print("Recommendation: Use Approach 2 for production systems!")
print()

EXPERIMENT 3: System vs User Prompt Placement

APPROACH 1: All CO-STAR in User Prompt
------------------------------------------------------------
# Your Beginner Workout Plan - You've Got This! 🌟

Hey there, future fitness superstar! I'm so proud of you for taking this first step. Remember, *everyone* starts somewhere, and today is YOUR day one. Let's keep it simple and fun!

---

## **Exercise 1: Bodyweight Squats**
- **Sets:** 2 sets
- **Reps:** 8-10 reps
- **Rest:** 60 seconds between sets

*Think of sitting back into a chair. You're building those leg muscles - you've got this!*

---

## **Exercise 2: Wall Push-Ups**
- **Sets:** 2 sets  
- **Reps:** 8-10 reps
- **Rest:** 60 seconds between sets

*Stand facing a wall and push away. This is perfect for building upper body strength at YOUR pace!*

---

## **Exercise 3: Marching in Place**
- **Sets:** 2 sets
- **Duration:** 30 seconds each
- **Rest:** 30 


APPROACH 2: Split - System (C,S,T,A,R) + User (O)
----------------------------

---
## 📋 Real-World CO-STAR Examples

Let's look at production-quality prompts for common use cases.

In [5]:
# Real-World CO-STAR Examples
print("=" * 60)
print("REAL-WORLD CO-STAR EXAMPLES")
print("=" * 60)
print()

examples = {
    "Code Review": {
        "context": "You are a senior software engineer reviewing Python code for a data processing pipeline.",
        "objective": "Identify bugs, performance issues, and suggest improvements following PEP 8 standards.",
        "style": "technical and precise",
        "tone": "constructive and educational",
        "audience": "mid-level Python developer",
        "response_format": "structured feedback with: Issues Found, Suggested Fixes, Best Practices"
    },
    
    "Customer Email": {
        "context": "You are a customer support specialist for a SaaS company.",
        "objective": "Respond to a customer complaint about billing discrepancies with empathy and a clear resolution path.",
        "style": "professional and warm",
        "tone": "empathetic and solution-focused",
        "audience": "frustrated small business owner",
        "response_format": "email format with greeting, acknowledgment, solution steps, and closing"
    },
    
    "Data Analysis": {
        "context": "You are a business analyst examining quarterly sales data for an e-commerce company.",
        "objective": "Summarize key trends and provide actionable insights to improve Q4 performance.",
        "style": "analytical and data-driven",
        "tone": "objective and professional",
        "audience": "C-level executives with limited time",
        "response_format": "executive summary with: Key Metrics, Trends, Recommendations (max 3 bullet points each)"
    },
    
    "Educational Content": {
        "context": "You are a computer science instructor teaching recursion to undergraduate students.",
        "objective": "Explain how recursion works with a clear example and common pitfalls to avoid.",
        "style": "educational and clear",
        "tone": "patient and encouraging",
        "audience": "second-year CS students with basic programming knowledge",
        "response_format": "tutorial format with: Concept Explanation, Code Example, Common Mistakes, Practice Problem"
    }
}

for use_case, params in examples.items():
    print(f"📌 Use Case: {use_case}")
    print("=" * 60)
    
    prompt = COSTARTemplate.build(**params)
    
    print("CO-STAR Prompt:")
    print(prompt)
    print()
    
    # Estimate cost
    estimated = estimate_tokens(prompt)
    print(f"Estimated input tokens: {estimated}")
    print(f"Estimated cost (with 200 token response): ${estimate_cost(prompt, 200, client.default_model):.6f}")
    print()
    print()

print("💡 These are production-ready prompt templates!")
print("   Save them and reuse for similar tasks.")
print()

REAL-WORLD CO-STAR EXAMPLES

📌 Use Case: Code Review
CO-STAR Prompt:
# Context
You are a senior software engineer reviewing Python code for a data processing pipeline.

# Objective
Identify bugs, performance issues, and suggest improvements following PEP 8 standards.

# Style
technical and precise

# Tone
constructive and educational

# Audience
mid-level Python developer

# Response Format
structured feedback with: Issues Found, Suggested Fixes, Best Practices


Estimated input tokens: 99
Estimated cost (with 200 token response): $0.003297


📌 Use Case: Customer Email
CO-STAR Prompt:
# Context
You are a customer support specialist for a SaaS company.

# Objective
Respond to a customer complaint about billing discrepancies with empathy and a clear resolution path.

# Style
professional and warm

# Tone
empathetic and solution-focused

# Audience
frustrated small business owner

# Response Format
email format with greeting, acknowledgment, solution steps, and closing


Estimated input t

---
## 🎯 Your Turn: Practice Tasks

Time to build your own CO-STAR prompts!

### 📝 Task 1: Build a CO-STAR Prompt from Scratch

**Scenario:** Choose one of these or create your own:
1. Writing a product description for an e-commerce site
2. Explaining a complex technical concept
3. Creating a social media post
4. Drafting a performance review
5. Summarizing a research paper

**Goal:** Build a complete CO-STAR prompt and test it.

In [6]:
# TODO - Task 1: Build Your CO-STAR Prompt
print("=" * 60)
print("TASK 1: Build Your Own CO-STAR Prompt")
print("=" * 60)
print()

# ============================================================================
# TODO: Fill in all 6 components
# ============================================================================

my_costar = COSTARTemplate.build(
    context="""
    # YOUR CONTEXT HERE
    # Example: You are a marketing copywriter for a sustainable fashion brand.
    """,
    
    objective="""
    # YOUR OBJECTIVE HERE
    # Example: Write a compelling product description for organic cotton t-shirts.
    """,
    
    style="""
    # YOUR STYLE HERE
    # Example: creative and engaging, suitable for social media
    """,
    
    tone="""
    # YOUR TONE HERE
    # Example: enthusiastic and eco-conscious
    """,
    
    audience="""
    # YOUR AUDIENCE HERE
    # Example: environmentally-aware millennials and Gen Z shoppers
    """,
    
    response_format="""
    # YOUR FORMAT HERE
    # Example: 2-3 short paragraphs, max 150 words total
    """
)

print("Your CO-STAR Prompt:")
print("=" * 60)
print(my_costar)
print("=" * 60)
print()

# Test your prompt
print("Testing your CO-STAR prompt...")
print()

response = client.generate(
    prompt=my_costar,
    temperature=0.7,
    max_tokens=300
)

if "error" not in response:
    print("✅ Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    print()
    print(f"📊 Tokens: {response['usage']['input_tokens']} in, {response['usage']['output_tokens']} out")
    
    tracker.add_call(response)
    
    print()
    print("=" * 60)
    print("REFLECTION")
    print("=" * 60)
    print()
    
    # ========================================================================
    # TODO: Reflect on your CO-STAR prompt
    # ========================================================================
    
    reflection = """
### What scenario did you choose?

[Describe your use case]

### How did each CO-STAR component help?

**Context:** [How did this help the LLM understand the task?]

**Objective:** [Was your goal clear and specific?]

**Style:** [Did the response match your desired style?]

**Tone:** [Was the emotional quality appropriate?]

**Audience:** [Was the complexity level right for your audience?]

**Response Format:** [Did you get the structure you wanted?]

### Quality of the response

[Rate 1-5 and explain]: ___/5

[What worked well?]

[What would you improve?]

### Real-world application

[Where would you use this prompt in practice?]
"""
    
    print(reflection)
    
    # Save reflection
    append_to_reflection(
        notebook="03",
        section_title="Task 1 - Build Your CO-STAR Prompt",
        reflection_content=reflection,
        output_dir=os.path.join(parent_dir, 'outputs')
    )
    
    print()
    print("💾 Reflection saved to outputs/homework_reflection.md")
    
else:
    print(f"❌ Error: {response['error']}")

print()

TASK 1: Build Your Own CO-STAR Prompt

Your CO-STAR Prompt:
# Context

    # YOUR CONTEXT HERE
    # Example: You are a marketing copywriter for a sustainable fashion brand.
    

# Objective

    # YOUR OBJECTIVE HERE
    # Example: Write a compelling product description for organic cotton t-shirts.
    

# Style

    # YOUR STYLE HERE
    # Example: creative and engaging, suitable for social media
    

# Tone

    # YOUR TONE HERE
    # Example: enthusiastic and eco-conscious
    

# Audience

    # YOUR AUDIENCE HERE
    # Example: environmentally-aware millennials and Gen Z shoppers
    

# Response Format

    # YOUR FORMAT HERE
    # Example: 2-3 short paragraphs, max 150 words total
    


Testing your CO-STAR prompt...

✅ Response:
# PROMPT TEMPLATE GUIDE

This is a structured template for creating clear, effective prompts. Fill in each section below:

---

## **Context**
*Who are you? What's the situation or background?*

Example: "You are a marketing copywriter for a sustain

### 📝 Task 2: CO-STAR Variations

Test how changing ONE component affects the response.

**Goal:** See the impact of individual components.

In [7]:
# TODO - Task 2: Component Variation Analysis
print("=" * 60)
print("TASK 2: CO-STAR Component Variation")
print("=" * 60)
print()

# ============================================================================
# TODO: Create a base CO-STAR prompt
# ============================================================================

base_params = {
    "context": "You are a travel guide writer.",
    "objective": "Describe a day trip to a mountain hiking trail.",
    "style": "descriptive and informative",
    "tone": "enthusiastic",
    "audience": "outdoor enthusiasts",
    "response_format": "2-3 paragraphs"
}

# ============================================================================
# TODO: Choose ONE component to vary (e.g., tone or audience)
# Then create 3 variations
# ============================================================================

component_to_vary = "tone"  # Change this to: style, tone, audience, or response_format

variations = {
    "Variation 1": "enthusiastic and excited",
    "Variation 2": "calm and zen-like", 
    "Variation 3": "adventurous and daring"
}

print(f"Testing variations of: {component_to_vary}")
print()

results = {}

for var_name, var_value in variations.items():
    print(f"📝 {var_name}: {var_value}")
    print("-" * 60)
    
    # Create modified params
    test_params = base_params.copy()
    test_params[component_to_vary] = var_value
    
    # Build prompt
    prompt = COSTARTemplate.build(**test_params)
    
    # Generate
    response = client.generate(
        prompt=prompt,
        temperature=0.7,
        max_tokens=200
    )
    
    if "error" not in response:
        print(response['content'])
        results[var_name] = response['content']
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

# ========================================================================
# TODO: Analyze the differences
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = f"""
### Component Varied: {component_to_vary}

### Base Configuration
{json.dumps(base_params, indent=2)}

### Variations Tested
{json.dumps(variations, indent=2)}

### Analysis

**Variation 1 ({variations['Variation 1']}):**
[How did this affect the response? What changed?]

**Variation 2 ({variations['Variation 2']}):**
[How did this differ from Variation 1?]

**Variation 3 ({variations['Variation 3']}):**
[How did this compare to the others?]

### Key Insight

[What did you learn about the {component_to_vary} component?]

### Best Choice

**For this use case, I would choose:** [Variation 1/2/3]

**Because:** [Your reasoning]
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="03",
    section_title="Task 2 - Component Variation Analysis",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 2: CO-STAR Component Variation

Testing variations of: tone

📝 Variation 1: enthusiastic and excited
------------------------------------------------------------
# A Day Trip to Summit Ridge Trail

Picture this: you're standing at the trailhead as the morning sun breaks through the pine canopy, casting golden streaks across the forest floor. Summit Ridge Trail beckons with 8.5 miles of pure alpine adventure, promising sweeping vistas, wildflower meadows, and that incredible sense of accomplishment that only comes from conquering a challenging ascent. The trail begins with a moderate climb through old-growth forest, where the scent of Douglas fir fills your lungs and the sound of rushing creek water accompanies your steady pace. After about two miles, the terrain opens up dramatically – suddenly you're above the treeline, where the real magic begins!

The final push to the ridge is where this hike truly shines. Switchbacks carved into the mountainside lead you through fields of Ind

### 📝 Task 3: Production Prompt Template

Create a reusable CO-STAR template for a real task you might encounter.

**Goal:** Build a production-ready prompt you can actually use.

In [8]:
# TODO - Task 3: Create a Production Template
print("=" * 60)
print("TASK 3: Production Prompt Template")
print("=" * 60)
print()

# ============================================================================
# TODO: Think of a real task you do regularly or might need to do
# Examples:
# - Writing emails to clients
# - Reviewing code
# - Creating social media posts
# - Analyzing data
# - Teaching/explaining concepts
# - Writing documentation
# ============================================================================

production_template = {
    "name": "My Production Template",  # Give it a descriptive name
    "use_case": "Describe when you'd use this",
    "parameters": {
        "context": """
        # YOUR CONTEXT
        """,
        "objective": """
        # YOUR OBJECTIVE
        """,
        "style": """
        # YOUR STYLE
        """,
        "tone": """
        # YOUR TONE
        """,
        "audience": """
        # YOUR AUDIENCE
        """,
        "response_format": """
        # YOUR FORMAT
        """
    }
}

print(f"Template Name: {production_template['name']}")
print(f"Use Case: {production_template['use_case']}")
print()

# Build the prompt
template_prompt = COSTARTemplate.build(**production_template['parameters'])

print("Full CO-STAR Prompt:")
print("=" * 60)
print(template_prompt)
print("=" * 60)
print()

# ============================================================================
# TODO: Create a sample user input to test with
# ============================================================================

sample_input = """
# YOUR SAMPLE INPUT HERE
# This should be a realistic example of what you'd use this template for
"""

# Combine template with sample input
full_prompt = f"{template_prompt}\n\n{sample_input}"

print("Testing with sample input...")
print()

response = client.generate(
    prompt=full_prompt,
    temperature=0.7,
    max_tokens=400
)

if "error" not in response:
    print("✅ Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    print()
    print(f"📊 Tokens: {response['usage']['input_tokens']} in, {response['usage']['output_tokens']} out")
    print(f"💰 Cost: ${estimate_cost(full_prompt, response['usage']['output_tokens'], client.default_model):.6f}")
    
    tracker.add_call(response)
    
    print()
    print("=" * 60)
    print("REFLECTION & TEMPLATE DOCUMENTATION")
    print("=" * 60)
    print()
    
    # ========================================================================
    # TODO: Document your template
    # ========================================================================
    
    reflection = f"""
### Template Details

**Name:** {production_template['name']}

**Use Case:** {production_template['use_case']}

**When to use this template:**
[Describe specific scenarios where this template would be helpful]

### CO-STAR Components Explained

**Context:** [Why did you choose this context?]

**Objective:** [What makes this objective clear and actionable?]

**Style:** [Why is this style appropriate?]

**Tone:** [How does this tone help achieve your objective?]

**Audience:** [Who is this tailored for and why?]

**Response Format:** [Why this structure?]

### Test Results

**Quality of output:** ___/5

**What worked well:**
[Your analysis]

**What could be improved:**
[Your analysis]

### Reusability Assessment

**How reusable is this template?** [High/Medium/Low]

**What would need to change for different inputs?**
[Your analysis]

**Estimated ROI:**
- Time saved per use: [X minutes]
- Frequency of use: [X times per week/month]
- Total time saved: [Calculate]

### Production Readiness

**Is this ready for real use?** [Yes/No/Needs refinement]

**Next steps to make it production-ready:**
1. [Action item]
2. [Action item]
3. [Action item]

### Template Storage

I will save this template:
- [ ] In a personal prompt library
- [ ] In team documentation
- [ ] In project codebase
- [ ] Other: ___________
"""
    
    print(reflection)
    
    # Save reflection
    append_to_reflection(
        notebook="03",
        section_title="Task 3 - Production Template",
        reflection_content=reflection,
        output_dir=os.path.join(parent_dir, 'outputs')
    )
    
    # Also save the template itself as JSON
    template_file = os.path.join(parent_dir, 'outputs', 'my_costar_template.json')
    with open(template_file, 'w') as f:
        json.dump(production_template, f, indent=2)
    
    print()
    print("💾 Reflection saved to outputs/homework_reflection.md")
    print(f"💾 Template saved to outputs/my_costar_template.json")
    print()
    print("💡 You can now reuse this template in real projects!")
    
else:
    print(f"❌ Error: {response['error']}")

print()

TASK 3: Production Prompt Template

Template Name: My Production Template
Use Case: Describe when you'd use this

Full CO-STAR Prompt:
# Context

        # YOUR CONTEXT
        

# Objective

        # YOUR OBJECTIVE
        

# Style

        # YOUR STYLE
        

# Tone

        # YOUR TONE
        

# Audience

        # YOUR AUDIENCE
        

# Response Format

        # YOUR FORMAT
        


Testing with sample input...

✅ Response:
# Context
I notice you've shared a template structure with placeholder sections, but the actual content for each section is missing. This appears to be a prompt engineering template designed to help structure AI requests effectively.

# Objective
To help you complete this template with actual content so it can be used for its intended purpose.

# Style
Clear, practical, and instructional

# Tone
Helpful and collaborative

# Audience
Someone learning to create effective AI prompts

# Response Format
Structured guidance with examples

---

## How to U

---
## 🎓 CO-STAR Best Practices

Let's review key lessons and best practices.

In [9]:
# CO-STAR Best Practices
print("=" * 60)
print("CO-STAR BEST PRACTICES")
print("=" * 60)
print()

best_practices = {
    "Context": [
        "✓ Provide role/expertise when relevant",
        "✓ Include domain-specific background",
        "✓ Set the scenario clearly",
        "✗ Don't include unnecessary details",
        "✗ Avoid contradictory information"
    ],
    
    "Objective": [
        "✓ Be specific and measurable",
        "✓ Use action verbs (analyze, create, summarize)",
        "✓ State success criteria when possible",
        "✗ Don't be vague or ambiguous",
        "✗ Avoid multiple conflicting objectives"
    ],
    
    "Style": [
        "✓ Match the medium (email, blog, report)",
        "✓ Consider formality level",
        "✓ Specify technical depth",
        "✗ Don't contradict tone",
        "✗ Avoid style-tone confusion"
    ],
    
    "Tone": [
        "✓ Align with audience expectations",
        "✓ Match the situation (serious vs casual)",
        "✓ Be consistent throughout",
        "✗ Don't use inappropriate tone",
        "✗ Avoid mixed signals"
    ],
    
    "Audience": [
        "✓ Specify expertise level",
        "✓ Consider their goals/needs",
        "✓ Think about their constraints (time, attention)",
        "✗ Don't assume too much knowledge",
        "✗ Avoid being too simplistic for experts"
    ],
    
    "Response Format": [
        "✓ Be very specific (JSON schema, markdown structure)",
        "✓ Specify length/word count if important",
        "✓ Define structure clearly",
        "✗ Don't leave format ambiguous",
        "✗ Avoid conflicting formatting requirements"
    ]
}

for component, practices in best_practices.items():
    print(f"🎯 {component}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("💡 GOLDEN RULES")
print("=" * 60)
print()
print("1. Not every prompt needs all 6 components")
print("   → Use what's relevant for your task")
print()
print("2. Response Format is often the most impactful")
print("   → Being specific about structure = better results")
print()
print("3. System prompts are great for C, S, T, A, R")
print("   → Reuse them across similar requests")
print()
print("4. Test and iterate")
print("   → First draft is rarely perfect")
print()
print("5. Save your best prompts")
print("   → Build a personal/team library")
print()


CO-STAR BEST PRACTICES

🎯 Context
------------------------------------------------------------
  ✓ Provide role/expertise when relevant
  ✓ Include domain-specific background
  ✓ Set the scenario clearly
  ✗ Don't include unnecessary details
  ✗ Avoid contradictory information

🎯 Objective
------------------------------------------------------------
  ✓ Be specific and measurable
  ✓ Use action verbs (analyze, create, summarize)
  ✓ State success criteria when possible
  ✗ Don't be vague or ambiguous
  ✗ Avoid multiple conflicting objectives

🎯 Style
------------------------------------------------------------
  ✓ Match the medium (email, blog, report)
  ✓ Consider formality level
  ✓ Specify technical depth
  ✗ Don't contradict tone
  ✗ Avoid style-tone confusion

🎯 Tone
------------------------------------------------------------
  ✓ Align with audience expectations
  ✓ Match the situation (serious vs casual)
  ✓ Be consistent throughout
  ✗ Don't use inappropriate tone
  ✗ Avoid mix

---
## 📊 Cost Analysis: CO-STAR Impact

Let's see how CO-STAR affects token usage and costs.

In [10]:
# CO-STAR Cost Analysis
print("=" * 60)
print("CO-STAR COST ANALYSIS")
print("=" * 60)
print()

# Compare: No structure vs Full CO-STAR
simple_prompt = "Write a product description for running shoes"

costar_prompt = COSTARTemplate.build(
    context="You are an e-commerce copywriter for a sports equipment retailer",
    objective="Write a compelling product description that highlights key features and benefits",
    style="persuasive and energetic",
    tone="enthusiastic and motivating",
    audience="recreational runners looking for quality everyday trainers",
    response_format="3 paragraphs: Features, Benefits, Call-to-action"
)

print("SIMPLE PROMPT")
print("-" * 60)
print(simple_prompt)
simple_tokens = estimate_tokens(simple_prompt)
print(f"Estimated tokens: {simple_tokens}")
print()

print("CO-STAR PROMPT")
print("-" * 60)
print(costar_prompt)
costar_tokens = estimate_tokens(costar_prompt)
print(f"Estimated tokens: {costar_tokens}")
print()

print("=" * 60)
print("COST COMPARISON")
print("=" * 60)
print()

token_difference = costar_tokens - simple_tokens
cost_difference = estimate_cost(costar_prompt, 200, client.default_model) - estimate_cost(simple_prompt, 200, client.default_model)

print(f"Token overhead: {token_difference} tokens ({token_difference/simple_tokens*100:.0f}% increase)")
print(f"Cost overhead: ${cost_difference:.6f} per request")
print()
print("But consider:")
print("  • Higher quality output (less iteration needed)")
print("  • More consistent results")
print("  • Fewer tokens wasted on poor responses")
print("  • Time saved on revisions")
print()
print("💡 CO-STAR overhead is usually worth it for production use!")
print()

# Show current costs
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()
print()

CO-STAR COST ANALYSIS

SIMPLE PROMPT
------------------------------------------------------------
Write a product description for running shoes
Estimated tokens: 11

CO-STAR PROMPT
------------------------------------------------------------
# Context
You are an e-commerce copywriter for a sports equipment retailer

# Objective
Write a compelling product description that highlights key features and benefits

# Style
persuasive and energetic

# Tone
enthusiastic and motivating

# Audience
recreational runners looking for quality everyday trainers

# Response Format
3 paragraphs: Features, Benefits, Call-to-action

Estimated tokens: 94

COST COMPARISON

Token overhead: 83 tokens (755% increase)
Cost overhead: $0.000249 per request

But consider:
  • Higher quality output (less iteration needed)
  • More consistent results
  • Fewer tokens wasted on poor responses
  • Time saved on revisions

💡 CO-STAR overhead is usually worth it for production use!

YOUR COSTS THIS NOTEBOOK

💰 API COST 

---
## ✅ Notebook 03 Complete!

### Summary

You've mastered the CO-STAR framework! You now know:
- ✅ All 6 components and when to use them
- ✅ How to build structured prompts
- ✅ System vs user prompt strategies
- ✅ Real-world applications
- ✅ Best practices and common pitfalls
- ✅ Cost considerations

In [11]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Overall reflection on CO-STAR
# ============================================================================

reflection = """
### 1. How has CO-STAR changed your approach to prompting?

[Your answer]

### 2. Which component do you find most impactful and why?

[Your answer]

### 3. When would you skip CO-STAR and use a simpler prompt?

[Your answer]

### 4. How will you use CO-STAR in your work/projects?

[Your answer]

### 5. What was your biggest "aha!" moment in this notebook?

[Your answer]

### 6. Rate your CO-STAR confidence: ___/5

[Explain your rating and what would increase it]
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="03",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

print()
print("=" * 60)
print("✅ NOTEBOOK 03 COMPLETE!")
print("=" * 60)
print()
print("Progress: [███████████░░░░░░░░░] 38% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework ← YOU ARE HERE")
print("○ Notebook 04: Structured Outputs")
print("○ Notebook 05: Chain of Thought")
print("○ Notebook 06: Model Comparison")
print("○ Notebook 07: MCP Introduction")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/04_structured_outputs.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. How has CO-STAR changed your approach to prompting?

[Your answer]

### 2. Which component do you find most impactful and why?

[Your answer]

### 3. When would you skip CO-STAR and use a simpler prompt?

[Your answer]

### 4. How will you use CO-STAR in your work/projects?

[Your answer]

### 5. What was your biggest "aha!" moment in this notebook?

[Your answer]

### 6. Rate your CO-STAR confidence: ___/5

[Explain your rating and what would increase it]


💾 Reflection saved to outputs/homework_reflection.md

✅ NOTEBOOK 03 COMPLETE!

Progress: [███████████░░░░░░░░░] 38% Complete

✓ Notebook 00: Setup Verification
✓ Notebook 01: Environment Setup
✓ Notebook 02: LLM Basics
✓ Notebook 03: CO-STAR Framework ← YOU ARE HERE
○ Notebook 04: Structured Outputs
○ Notebook 05: Chain of Thought
○ Notebook 06: Model Comparison
○ Notebook 07: MCP Introduction
○ Notebook 08: Project Kickoff

Next: notebooks/04_structured_outputs.ipynb

